<a href="https://colab.research.google.com/github/kairos0101/ChatBot_Projects/blob/main/ChatBot_DeepSeek_R1_Distill_Qwen_7B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get('HuggingFace')

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [ ]:
!pip install -U bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 1. Specify the model name from Hugging Face
# This specific model is a 7B parameter model distilled from DeepSeek-R1, based on Qwen2.5-Math-7B
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

# 2. Load the tokenizer and model
# Using device_map="cuda" automatically distributes the model across available GPUs
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="cuda", # Use "cpu" if a GPU is not available
    trust_remote_code=True
)

# 3. Create a text generation pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
# 4. Define a prompt
# The model is fine-tuned on samples generated by DeepSeek-R1 and performs well on reasoning tasks.
# A specific prompt format (like an instruction) may be more effective for optimal results.
prompt = "What is the capital of France? Provide a step-by-step reasoning."

# 5. Generate text
outputs = generator(
    prompt,
    max_length=200,          # Set a maximum length for the generated response
    temperature=0.6,         # Recommended temperature is between 0.5-0.7 for these models
    top_p=0.95,              # Recommended top_p value
    do_sample=True,          # Enables sampling (creative generation)
    repetition_penalty=1.1   # Recommended to prevent repetition, especially for non-English languages
)

# 6. Print the output
print("Generated Text:")
print(outputs[0]['generated_text'])

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# Specify the model name (from Hugging Face or a local path)
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

# Load the model and tokenizer
# device_map="auto" will use available GPUs
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    torch_dtype="auto",
    device_map="auto"
).eval()
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Create a text generation pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Define a function to generate text
def generate_text(prompt, max_length=256, temperature=0.7, top_p=0.9):
    outputs = generator(
        prompt,
        max_length=max_length,
        temperature=temperature,
        top_p=top_p,
        do_sample=True
    )
    return outputs[0]['generated_text']

In [ ]:
# Example prompt
prompt = "Write Python code to reverse a linked list"
output_text = generate_text(prompt)
print("Generated Text:\n", output_text)

In [ ]:
# Example prompt
prompt = "What is the capital of France? Provide a step-by-step reasoning."
output_text = generate_text(prompt)
print("Generated Text:\n", output_text)

In [ ]:
# Example prompt
prompt = "Give me a short introduction to AI computer vision."
output_text = generate_text(prompt)
print("Generated Text:\n", output_text)